# EDA учебной базы данных afisha

**Задача.** Предварительный анализ базы `data-analyst-afisha` (5 таблиц) по полному заданию:

1. **Взаимосвязи таблиц** — первичные (PK) и внешние (FK) ключи, типы связей (1:1, 1:M, M:N).
2. **Содержимое таблиц** — соответствие описанию, объём данных.
3. **Корректность данных** — уникальность идентификаторов, пропуски, написание категориальных значений (устройства, города, регионы, валюты).
4. **Распределение заказов по категориям** — равномерность, доминирующие и малочисленные категории.
5. **Аномалии в выручке** — статистика, выбросы, особенности.
6. **Период и сезонность** — за какой период данные, возможны ли сезонные эффекты.

Основной фокус — таблицы `purchases` и `events`: активность клиентов, типы мероприятий, выручка сервиса.

**Инструменты:** PostgreSQL через DBeaver (основной), Jupyter через VS Code.

In [6]:
from getpass import getpass
from sqlalchemy import create_engine

%load_ext sql

password = getpass("Пароль: ")

engine = create_engine(
    "postgresql+psycopg2://praktikum_student@"
    "rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha",
    connect_args={"password": password, "sslmode": "require"},
    pool_pre_ping=True,
)

%sql engine

## Этап 1. Схема: ключи и типы связей

По описанию схемы вырисовывается такая картина ключей:

| Таблица | PK (первичный ключ) | FK (внешние ключи) |
|---|---|---|
| purchases | order_id | event_id → events |
| events | event_id | city_id → city, venue_id → venues |
| city | city_id | region_id → regions |
| venues | venue_id | — |
| regions | region_id | — |

Типы связей: **все связи «один ко многим» (1:M)**. Связей «один к одному» нет (ни одна таблица не расширяет другую строка-в-строку), «многие ко многим» тоже нет.

```
regions —< city —< events —< purchases
venues  —< events
```

**Вывод этапа.** основная таблица — `purchases`, измерения подтягиваются JOIN'ами через `events`.

## Этап 2. Проверка ключей самими данными

**Что и зачем.** Описание схемы может расходиться с реальностью. Тип ключа проверяется простым приёмом: сравнением `COUNT(*)` и `COUNT(DISTINCT колонка)` внутри конкретной таблицы:
- числа равны → колонка уникальна в этой таблице → ведёт себя как PK;
- `DISTINCT` заметно меньше → значения повторяются → колонка здесь FK, и связь с родительской таблицей 1:M.

**Проверка 1: `order_id` в purchases — ключ?**

In [ ]:
SELECT COUNT(*)               AS total_rows,
       COUNT(order_id)        AS non_null_ids,
       COUNT(DISTINCT order_id) AS unique_ids
FROM afisha.purchases;

**Результат:** 292 034 = 292 034 = 292 034 — три числа совпадают.

**Проверка 2: `event_id` в purchases — уникален или повторяется?**

In [ ]:
SELECT COUNT(*)                  AS total_rows,
       COUNT(DISTINCT event_id) AS unique_events
FROM afisha.purchases;

**Результат:** 292 034 строки, но только 22 484 уникальных `event_id` → в среднем ≈ 13 покупок на мероприятие. Таблица `events` содержит также 22 484 уникальных `event_id`

**Вывод этапа.** Заявленная схема подтвердилась данными:
- `order_id` — полноценный PK таблицы `purchases`: один заказ = одна строка (покупки не «разбиты» на позиции, `SUM` по заказу не нужен);
- `event_id` в purchases — FK: связь `events` → `purchases` действительно «один ко многим», ≈13 заказов на мероприятие.

## Этап 3. Объём данных и период

**Объём таблиц:**

| Таблица | Строк |
|---|---|
| purchases (заказы) | 292 034 |
| events (мероприятия) | 22 484 |
| city (города) | 353 |
| regions (регионы) | 81 |
| уникальные пользователи в purchases | 22 000 |

In [ ]:
SELECT MIN(created_dt_msk),
       MAX(created_dt_msk),
       COUNT(DISTINCT created_dt_msk) AS unique_created_dates,
       MAX(created_dt_msk) - MIN(created_dt_msk) AS date_range
FROM afisha.purchases;

**Результат:** с 2024-06-01 по 2024-10-31, 153 уникальные даты при интервале 152 дня (152 + 1, обе границы включены).

**Вывод этапа.** Данные покрывают 5 месяцев (июнь–октябрь 2024), причём **без единого пропуска по дням** — есть заказы за каждые сутки периода. Объём солидный: ~2 тыс. заказов в день, 22 тыс. пользователей, география — 81 регион.

Ограничение для интерпретаций: сезон летних фестивалей и начала театрального сезона; выводы о годовом цикле на этих данных делать нельзя.

## Этап 4. Пропуски (NULL)

**Автоматизировано.**  запрос-генератор собирает их автоматически из служебной таблицы `information_schema.columns`, где СУБД хранит список всех колонок.

In [ ]:
-- Шаг 1: генератор. Собирает единую проверку по всем колонкам всех таблиц
SELECT string_agg(
    format(
        'SELECT %I AS table_name, %L AS column_name, COUNT(*) - COUNT(%I) AS null_count FROM %I.%I',
        table_name, column_name, column_name, table_schema, table_name
    ),
    ' UNION ALL '
)
FROM information_schema.columns
WHERE table_schema = 'afisha'
  AND table_name IN ('purchases', 'events', 'venues', 'city', 'regions');

-- Шаг 2: выполнить текст из ячейки результата — одна большая UNION-проверка

**Результат:** нет ни одного пропуска ни в одной колонке всех пяти таблиц.

**Вывод этапа.** База без пропусков

## Этап 5. Категориальные значения

**Задача:** Необходимо проверить «написание» категориальных данных: опечатки и нестандартные значения.

**Результаты по всем категориальным полям:**

| Поле | Значений | Наблюдения |
|---|---|---|
| cinema_circuit | 6 | «нет» — 290 632 (99%+); сети: Другое 1 263, КиноСити 124, Киномакс 7, Москино 7, ЦентрФильм 1. Почти все мероприятия вне кинотеатров |
| currency_code | 2 | rub — 286 961; kzt — 5 073 |
| device_type_canonical | 5 | mobile — 232 679 (~80%); desktop — 58 170; tablet — 1 180; other — 2; tv — 3 |
| age_limit | 5 | только допустимые значения: 0 / 6 / 12 / 16 / 18 |
| service_name | 36 | 36 билетных операторов |

**Вывод этапа.** Все категориальные поля — в «каноническом» виде: опечаток, синонимов и мусорных значений нет.

Два наблюдения для будущей аналитики:
1. **Две валюты** — суммы `total`/`revenue` нельзя агрегировать в одну цифру без разбивки по валюте (тенге и рубли в одном SUM — бессмысленное число);
2. **Mobile-first** — 80% заказов с мобильных устройств: пользовательская аналитика должна начинаться с mobile.

Пример запроса-ревизии (остальные поля проверялись аналогично):

In [ ]:
SELECT device_type_canonical, COUNT(*)
FROM afisha.purchases
GROUP BY device_type_canonical
ORDER BY COUNT(*) DESC;

## Этап 6. Уникальность идентификаторов и дубликаты

**Что и зачем.** Дубликаты бывают двух видов: полные (одна строка скопирована) — грязь, и «частичные» (строки различаются лишь служебным id) — возможная бизнес-логика. Полных дублей по всем таблицам нет (`duplicated`-проверка). Но в `events` обнаружилось подозрительное: строк больше, чем уникальных комбинаций содержательных полей.

**Проверка:** группировка по всем колонкам, кроме `event_id`, через `ROW_NUMBER()`:

In [ ]:
SELECT COUNT(*) AS duplicate_rows
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY event_name_code, event_type_description,
                              event_type_main, organizers, city_id, venue_id
                              ORDER BY event_id) AS rn
    FROM afisha.events
) t
WHERE rn > 1;

**Результат:** 873 строки идентичны по всем колонкам, кроме `event_id`.

**Вывод этапа.** В справочнике `events` одно и то же мероприятие может существовать под несколькими `event_id` — вероятная причина: одинаковые мероприятия, проходившие в разные даты (или дубль при загрузке справочника). Практическое следствие для аналитики: **мероприятие нельзя считать единицей, равной `event_id`** — при подсчёте «сколько мероприятий» нужно группировать по содержательной комбинации полей.

## Этап 7. Расследование аномалии: 5 772 заказа с revenue = 0

**Задача** ≈2% заказов имеют нулевую выручку. Это либо особенность бизнеса (бесплатные события), либо дефект данных (отмены, потерянные суммы). Цепочка из четырёх шагов, где каждый следующий уточняет гипотезу, оставшуюся после предыдущего.

### Шаг 1. Есть ли у zero-заказов билеты?

**Гипотеза:** если билеты не выписаны — «пустые»/отменённые заказы; если выписаны — заказ живой.



In [4]:
%%sql
SELECT tickets_count, COUNT(*)
FROM afisha.purchases
WHERE revenue = 0
GROUP BY tickets_count
ORDER BY tickets_count;
-- Результат: все 5772 заказа имеют tickets_count >= 1 (в основном 1-6, выбросы до 57)


Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

10 rows affected.

tickets_count,count
1,965
2,1815
3,1867
4,981
5,114
6,24
17,2
19,1
30,2
57,1


**Результат:** все 5 772 заказа имеют `tickets_count ≥ 1`.

**Вывод шага:** заказы «живые» — гипотеза «отменённые пустые заказы» исключена.

### Шаг 2. Нулевая выручка — свойство заказа или мероприятия?

**Гипотеза:** если zero-заказы размазаны по мероприятиям с другой выручкой — возвраты/промо на уровне заказов; если нулевая выручка у мероприятий целиком — бесплатные события.



In [6]:
%%sql
SELECT
    COUNT(*) AS zero_events,
    SUM(flag_0) AS events_with_mixed_revenue
FROM (
    SELECT p.event_id,
           MAX(CASE WHEN p.revenue > 0 THEN 1 ELSE 0 END) AS flag_0
    FROM afisha.purchases p
    GROUP BY p.event_id
    HAVING SUM(p.revenue) = 0
) t;
-- Результат: 295 | 0


Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

1 rows affected.

zero_events,events_with_mixed_revenue
295,0


**Результат:** **295 мероприятий** с нулевой суммарной выручкой, и **ни у одного нет ни одного платного заказа** (`events_with_mixed_revenue = 0`).

**Вывод шага:** нулевая выручка — свойство **мероприятия целиком**. «Смешанных» мероприятий нет; остаётся главная гипотеза — бесплатные события.

Приём этапа: `MAX(CASE WHEN условие THEN 1 ELSE 0 END)` — «условие выполнилось хотя бы раз в группе». Он страхует от ложной интерпретации: нулевая сумма может получиться и взаимным погашением платёж + возврат. Поэтому шаг 3 обязателен.

### Шаг 3. Может ли возврат «погасить» мероприятие до нуля?

**Гипотеза:** если есть отрицательные выручки (возвраты), мероприятие из шага 2 могло бы быть платным со стопроцентным возвратом. Смотрим минимумы:



In [7]:
%%sql
SELECT MIN(revenue), MIN(total)
FROM afisha.purchases;
-- Результат: -90.76 | -122.65


Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

1 rows affected.

min,min_1
-90.76,-358.85


**Результат:** самый крупный возврат в базе — комиссия 90.76 ₽ (при медианном чеке 4 826 ₽). Но главный аргумент уже дан шагом 2: у всех zero-мероприятий нет ни одной строки с `revenue > 0` — «погашение» продажи возвратом исключено напрямую

### Шаг 4. Контроль на выборке

Точечная проверка zero-мероприятий: у всех заказов `revenue = 0` и `total = 0` — нули по всей цепочке, включая сумму, заплаченную клиентом.

**Итог этапа.** **295 мероприятий базы — полностью бесплатные** (≈20 заказов на каждое, ≈2% всех заказов). Возвраты и отмены исключены: среди zero-мероприятий нет ни одной строки с положительной выручкой, а возвраты существуют как отдельные строки — и не встречаются среди zero-мероприятий. Статуса оплаты в схеме нет, поэтому «промо vs. бесплатные» средствами базы неразрешимы — честная граница исследования.



## Этап 8. Отрицательная выручка: паспорт минусовых заказов

**Задача.** Обнаружено: 381 заказ имеет `revenue < 0`. Природа минусов важна дважды: искажают ли они метрики выручки, и могут ли возвраты «погасить» платное мероприятие.

**Исходная гипотеза:** копеечные минуса — комиссия платёжного шлюза или корректировки биллинга, «технические» суммы, а не возвраты.

**Проверка:** «паспорт» — смотрим `revenue` вместе с `total`, `tickets_count`, валютой и оператором:



In [ ]:
SELECT p.revenue, p.total, p.tickets_count, p.currency_code, p.service_name
FROM afisha.purchases p
WHERE p.revenue < 0
ORDER BY p.revenue
LIMIT 20;


**Результат (первые 20, по величине минуса):**

| revenue | total | tickets | валюта | оператор | примечание |
|---|---|---|---|---|---|
| −90.76 | −122.65 | 5 | rub | Билеты в руки | ⚠️ доля 74% от total |
| −17.94 | −358.85 | 5 | rub | Билеты в руки | доля ≈ 5% |
| −10.77 | −215.31 | 3 | rub | Билеты в руки | ×8 одинаковых строк |
| −8.53 | −213.36 | 4 | rub | Лови билет! | ×2 одинаковые строки |
| −7.18 | −143.54 | 2 | rub | Билеты в руки | ×9 одинаковых строк |

**Три наблюдения, изменившие интерпретацию:**

1. **`total` тоже отрицательный** — главный факт: клиенту вернули деньги. Это записи **возвратов**, а не «комиссия шлюза». Исходная гипотеза опровергнута данными.
2. **`revenue` ≈ 5% от `total`** — возврат зеркалит исходный заказ: сервис возвращает клиенту всю сумму вместе со своим сбором. (Уточнение этапа 10: 5% — лишь один из тарифов сетки.)
3. **Операторы «Билеты в руки» и «Лови билет!»** — возвраты сконцентрированы у конкретных операторов (или артефакт сортировки по величине минуса — проверить доли по всем 381).
4. **Возвраты — история микро-заказов** (проверка распределения): медиана |total| возврата **38.76 ₽** против **4 826.49 ₽** у продаж — в 124 раза меньше; максимум |total| возврата — **358.85 ₽** при максимуме продаж 811 745 ₽. Ни один возврат в базе не дороже 359 ₽: дорогие заказы не возвращаются вовсе (или возвращаются вне этой механики). Гипотезы для бизнеса: дешёвые одиночные билеты возвращаются «кнопкой», групповые/дорогие — через ручной процесс вне системы.

**Аномалия внутри аномалии:** строка −90.76 при `total` −122.65 — доля 74%, а не ~5%: частичный возврат или удержание при возврате.

**Одинаковые строки** (×8, ×9 одинаковых сумм) — вероятно, по-билетные возвраты одного заказа или одинаковые заказы одного мероприятия; полных дублей в purchases нет (этап 6).

**Вывод этапа.** Минусовая выручка — штатная механика возвратов: строка-зеркало с отрицательными `total` и `revenue`. При агрегации выручки эти строки учитывать отдельно: они уменьшают суммы продаж.



## Этап 9. Распределение заказов по категориям ⬜

**Задача.** Проверить равномерность распределения заказов: доминирующие и малочисленные категории определяют, где метрики надёжны, а где нестабильны.

**Что уже известно (и говорит о неравномерности):**

| Признак | Крупнейшая категория | Малые категории (⚠️) |
|---|---|---|
| device_type_canonical | mobile — ~80% заказов | other — 2, tv — 3 |
| currency_code | rub — 98.3% | kzt — 1.7% |
| cinema_circuit | нет — 99.6% | ЦентрФильм — 1 заказ |
| age_limit | 16+ (78 864), 12+ (62 861), 0+ (61 731) — относительно ровно | — |


## 9.2: главный разрез — тип мероприятия:

In [ ]:
%%sql
SELECT e.event_type_main,
       COUNT(*) AS orders,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS share_pct,
       SUM(p.total) FILTER (WHERE p.currency_code = 'rub') AS total_rub
FROM afisha.purchases p
JOIN afisha.events e ON p.event_id = e.event_id
GROUP BY e.event_type_main
ORDER BY orders DESC;



Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

8 rows affected.

event_type_main,orders,share_pct,revenue_rub
концерты,115634,39.6,945561600.0
театр,67744,23.2,432889730.0
другое,66109,22.6,265903020.0
спорт,22006,7.5,61173200.0
стендап,13424,4.6,84103016.0
выставки,4873,1.7,10914894.0
ёлки,2006,0.7,15006761.0
фильм,238,0.1,30556.12


**Результаты:** Распределение заказов по типам сильно неравномерное: концерты доминируют (39.6% заказов), далее театр (23.2%) и заметно «другое» (22.6%).

## Этап 10. Статистика выручки, выбросы и модель комиссии

**Задача.** Профиль выручки: типичные значения, разброс, выбросы — и природа самой метрики `revenue`. Аномалии `revenue = 0` (295 бесплатных мероприятий) и `revenue < 0` (возвраты) разобраны в этапах 7–8.

### 10.1. revenue: сервис комиссии, характеристики

Распределение `revenue / total` по всем заказам с продажами:



In [34]:
%%sql
WITH perc AS (
    SELECT ROUND((p.revenue * 100.0 / p.total)::numeric, 0) AS percent
    FROM afisha.purchases p
    WHERE p.revenue > 0
)
SELECT  percent, 
        COUNT(*) AS orders,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS share_pct
FROM perc
GROUP BY percent
ORDER BY orders DESC;


Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

24 rows affected.

percent,orders,share_pct
4,34503,12.07
9,33170,11.60
8,26150,9.15
3,25431,8.90
11,25283,8.84
5,23787,8.32
10,23745,8.31
12,20413,7.14
7,16553,5.79
13,14350,5.02


**Результат (очищенная версия — джиттер убран округлением до целого тарифа):** лестница из **24 целых тарифов** — плотная зона 1–22% и экзотика 32/34%. Сумма долей = 100%, дробного шума больше нет.

| Тариф, % | Заказов | Доля |
|---|---|---|
| 4 | 34 503 | 12.07% |
| 9 | 33 170 | 11.60% |
| 8 | 26 150 | 9.15% |
| 3 | 25 431 | 8.90% |
| 11 | 25 283 | 8.84% |
| 5 | 23 787 | 8.32% |
| 10 | 23 745 | 8.31% |
| 12 | 20 413 | 7.14% |
| 7 | 16 553 | 5.79% |
| 13 | 14 350 | 5.02% |
| 6 | 13 237 | 4.63% |
| 2 | 12 838 | 4.49% |
| 14–17 | 15 528 | 5.43% |
| остальные (1, 18–22, 32, 34) | 893 | 0.31% |

Чтение распределения:

- **Рабочая зона 2–13% держит 94.3% всех заказов.** Вершина плоская: 4%, 9%, 8%, 3%, 11% идут с долями 8.8–12.1% — единого «стандартного» тарифа нет, тарифная политика диверсифицирована.
- **Средневзвешенный тариф ≈ 7.9%** — эффективная комиссия сервиса по всем заказам с продажами.
- **14–17% — 5.4%** заказов; **экзотика 18–34% — ~440 заказов (0.16%)**, включая одиночные тарифы 32% и 34%.

## 10.2 Формирование ставки. Оператор или мероприятие
**Кто задаёт ставку?** Кандидат 1 — оператор:


In [36]:
%%sql
SELECT p.service_name,
       COUNT(*) AS orders,
       ROUND((MIN(p.revenue * 100.0 / NULLIF(p.total, 0)))::numeric, 2) AS min_pct,
       ROUND((AVG(p.revenue * 100.0 / NULLIF(p.total, 0)))::numeric, 2) AS avg_pct,
       ROUND((MAX(p.revenue * 100.0 / NULLIF(p.total, 0)))::numeric, 2) AS max_pct
FROM afisha.purchases p
WHERE p.revenue > 0
GROUP BY p.service_name
ORDER BY orders DESC;


Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

36 rows affected.

service_name,orders,min_pct,avg_pct,max_pct
Билеты без проблем,58712,1.00,6.79,16.00
Лови билет!,41040,1.00,7.57,20.00
Билеты в руки,40021,2.00,6.77,34.00
Мой билет,34953,1.00,8.43,21.00
Облачко,26730,6.00,10.66,22.00
Лучшие билеты,17729,1.00,4.22,12.00
Весь в билетах,16910,6.00,10.48,18.00
Прачечная,10385,1.00,7.29,32.00
Край билетов,6238,5.00,10.86,18.00
Тебе билет!,5241,5.00,9.09,15.00


**Результат по 36 операторам:** ставка НЕ фиксирована на уровне оператора — у большинства разброс во всю сетку («Билеты без проблем» 1–16%, «Билеты в руки» 2–34%); фиксированные ставки — только у мелких («Кино билет» 9%, «Зе Бест!» 3%, «Билеты в интернете» 6%). Но **средние операторов различаются систематически**: от 3.5% («Радио ticket») до 12.9% («Восьмёрка»); крупные сидят в коридоре 6.7–8.4%.

**Гипотеза уточнена:** ставка назначается на уровне **мероприятия**. 

**Гипотеза** — фиксирован ли тариф внутри `event_id`: Смягчённая проверка: тарифы, округлённые до целого (допуск на копеечный джиттер)

In [50]:
%%sql
SELECT COUNT(*) AS events_total,
       SUM(CASE WHEN min_r = max_r THEN 1 ELSE 0 END) AS events_fixed_rate
FROM (
    SELECT p.event_id,
           MIN(ROUND((p.revenue * 100.0 / NULLIF(p.total, 0))::numeric, 0)) AS min_r,
           MAX(ROUND((p.revenue * 100.0 / NULLIF(p.total, 0))::numeric, 0)) AS max_r
    FROM afisha.purchases p
    WHERE p.revenue > 0
    GROUP BY p.event_id
) t;

Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

1 rows affected.

events_total,events_fixed_rate
22123,22123


**Результат:** При округлении до целого тарифа: **22 123 = 22 123 — у всех мероприятий тариф фиксирован.**

**Модель выручки сервиса:**

1. **Мероприятию назначается тариф** — целое число процентов из сетки 1–34%.
2. **Каждый заказ:** `revenue` = тариф × `total`, округлённый до копеек.
3. **Вероятно возврат** — строка-зеркало: `total < 0`, `revenue` = −(тариф × |total|).

**Замечание:** способ хранения данных — артефакт округления маскируется под бизнес-разброс значений, округления и типы данных дают разные величины сущностно идентичные.

### 10.2. Классификация мероприятий по выручке

Счётное тождество: 22 484 мероприятий в purchases = 22 123 с продажами + 295 бесплатных (этап 7) + **остаток 66**:



In [52]:
%%sql
SELECT
    SUM(CASE WHEN neg = orders THEN 1 ELSE 0 END) AS fully_refunded,   -- все заказы — возвраты
    SUM(CASE WHEN neg < orders THEN 1 ELSE 0 END) AS mixed_with_zero   -- есть и нулевые заказы
FROM (
    SELECT p.event_id,
           COUNT(*) AS orders,
           COUNT(*) FILTER (WHERE p.revenue < 0) AS neg
    FROM afisha.purchases p
    GROUP BY p.event_id
    HAVING SUM(p.revenue) < 0
) t;

Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

c:\Users\hitos\PycharmProjects\data-analyst-afisha\.venv\Lib\site-packages\sql\connection\connection.py:881: JupySQLRollbackPerformed: Current transaction is aborted. JupySQL executed a ROLLBACK operation.
  warnings.warn(


1 rows affected.

fully_refunded,mixed_with_zero
66,0


**Результат:** 66 — где все заказы отрицательны, т.е. полностью возвращены, скорее всего отмена мероприятия. Полная классификация мероприятий:

| Класс | Кол-во | Признак |
|---|---|---|
| платные | 22 123 | есть заказ с `revenue > 0` |
| бесплатные | 295 | `SUM = 0`, нет платных заказов (этап 7) |
| полностью возвращённые | 66 | `SUM < 0`, все заказы возвращены |
| **итого** | **22 484** | — |

66 «мероприятий-призраков» — билеты покупали, но каждый заказ возвращен. При метриках продаж исключать или разбирать отдельно (массовый возврат часто = отмена события).

### 10.3. Статистики чека



In [24]:
%%sql
SELECT MIN(total)     AS min_total,
       ROUND(AVG(total)::numeric, 2) AS mean_,
       PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY total) AS p25,
       PERCENTILE_CONT(0.5)  WITHIN GROUP (ORDER BY total) AS p50,
       MAX(total)     AS max_total
FROM afisha.purchases
WHERE revenue > 0 AND currency_code = 'rub';

Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

1 rows affected.

min_total,mean_,p25,p50,max_total
0.93,6465.25,2266.139892578125,4826.490234375,811745.4


**Результат (продажи, rub):**

| Метрика | Значение |
|---|---|
| MIN | 0.93 ₽ |
| p25 | 2 266.14 ₽ |
| **медиана (p50)** | **4 826.49 ₽** |
| среднее | 6 465.25 ₽ |
| MAX | 811 745.40 ₽ |

**Интерпретация:**

- **Среднее в 1.34 раза выше медианы** → распределение скошено вправо: «средний чек» без медианы вводил бы в заблуждение.
- **MAX в 168 раз выше медианы** → экстремальные выбросы существуют (811 745 ₽ — кандидат: групповая/корпоративная закупка; стоит посмотреть строку).
- **MIN = 0.93 ₽** — заказ на 93 копейки с положительной выручкой: микро-аномалия, кандидат на проверку.


## Этап 11. Сезонность активности

**Задача.** Динамика по месяцам. Период установлен (этап 3): 2024-06-01 → 2024-10-31, 153 дня без пропусков.

**Динамика по месяцам:**

In [7]:
%%sql
SELECT
	date_trunc('month', created_dt_msk) AS month_,
	count(*),
	SUM(revenue) FILTER (
	WHERE currency_code = 'rub') AS revenue_rub
FROM
	afisha.purchases p
GROUP BY 
	date_trunc('month', created_dt_msk);

# month_	count	revenue_rub
# 2024-06-01 00:00:00	34840	17917570.0
# 2024-07-01 00:00:00	41112	24061418.0
# 2024-08-01 00:00:00	45217	26539020.0
# 2024-09-01 00:00:00	70265	36334792.0
# 2024-10-01 00:00:00	100600	52278352.0


Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

5 rows affected.

month_,count,revenue_rub
2024-06-01 00:00:00,34840,17917566.0
2024-07-01 00:00:00,41112,24061436.0
2024-08-01 00:00:00,45217,26539074.0
2024-09-01 00:00:00,70265,36334760.0
2024-10-01 00:00:00,100600,52278376.0


In [8]:
%%sql
SELECT DATE_TRUNC('month', created_dt_msk) AS month_,
       COUNT(*)                                              AS orders,
       COUNT(*) FILTER (WHERE currency_code = 'rub')         AS orders_rub,
       ROUND((SUM(total)   FILTER (WHERE currency_code = 'rub'))::numeric, 0) AS turnover_rub,
       ROUND((SUM(revenue) FILTER (WHERE currency_code = 'rub'))::numeric, 0) AS revenue_rub,
       ROUND((AVG(total)   FILTER (WHERE currency_code = 'rub' AND revenue > 0))::numeric, 0) AS avg_check_rub
FROM afisha.purchases
GROUP BY 1
ORDER BY 1;

Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

5 rows affected.

month_,orders,orders_rub,turnover_rub,revenue_rub,avg_check_rub
2024-06-01 00:00:00,34840,32449,207859000,17917600,6705
2024-07-01 00:00:00,41112,40848,273705000,24061400,7146
2024-08-01 00:00:00,45217,44513,298544000,26539100,6925
2024-09-01 00:00:00,70265,69143,422658000,36334800,6155
2024-10-01 00:00:00,100600,100008,612812000,52278400,6144


**Результат:**

| Месяц | Заказов | Из них rub | Оборот rub, млн | Комиссия rub, млн | Средний чек rub (продажи) |
|---|---|---|---|---|---|
| июнь | 34 840 | 32 449 | 207.9 | 17.9 | 6 705 ₽ |
| июль | 41 112 | 40 848 | 273.7 | 24.1 | 7 146 ₽ |
| август | 45 217 | 44 513 | 298.5 | 26.5 | 6 925 ₽ |
| сентябрь | 70 265 | 69 143 | 422.7 | 36.3 | 6 155 ₽ |
| октябрь | 100 600 | 100 008 | 612.8 | 52.3 | 6 144 ₽ |

Итого за период: оборот rub ≈ **1.82 млрд ₽**, комиссия сервиса ≈ **157 млн ₽**, эффективная ставка ≈ **8.7%**.

**Три наблюдения:**

1. **Рост чисто объёмный:** заказы ×2.9 (34 840 → 100 600), комиссия ×2.9 (17.9 → 52.3 млн ₽). Эффективная ставка стабильна: 8.5–8.9% по месяцам — растёт база, а не наценка.
2. **Чек движется против объёма:** средний чек падает с летних 6 705–7 146 ₽ к осенним ~6 150 ₽ (−8…−14%). Лето — дорогие заказки, осень — массовый спрос дешевле: рост обеспечен количеством, а не подорожанием.
3. **Тенге-активность рваная:** kzt-заказы по месяцам — 2 391 / 264 / 704 / 1 122 / 592 (сумма 5 073, тождество с этапом 5 сошлось). 47% тенге-заказов — в июне: след одного-двух крупных тенге-событий, а не постоянного спроса.

**Оговорка:** 5 месяцев — тренд внутри периода, а не годовая сезонность: «осень горячее лета» — можно, выводы о годовом цикле — нельзя.

## Этап 12. Количество плательщиков в kzt

In [4]:
%%sql
SELECT
    *,
    ROUND((100.0 * users_with_kzt / users_total), 2)
FROM    (SELECT COUNT(DISTINCT user_id) AS users_total,
         COUNT(DISTINCT user_id) FILTER (WHERE currency_code = 'kzt') AS users_with_kzt
        FROM afisha.purchases);

UsageError: Cell magic `%%sql` not found.


 1362 kzt-пользователя (6.19%) делают 5 067 продаж → ~3.7 заказа на человека против ~13 у rub-аудитории.

## Этап 13. Мероприятия с оплатой в двух валютах

**Задача.** Проверить, есть ли мероприятия, принимающие оплату в двух валютах: флаги по обеим валютам в разрезе `event_id`.

In [56]:
%%sql
WITH cnt AS (
    SELECT event_id,
           MAX(CASE WHEN currency_code = 'rub' THEN 1 ELSE 0 END) AS flag_rub,
           MAX(CASE WHEN currency_code = 'kzt' THEN 1 ELSE 0 END) AS flag_kzt,
           SUM(total) FILTER (WHERE currency_code = 'rub') AS rub_total,
           SUM(total) FILTER (WHERE currency_code = 'kzt') AS kzt_total
    FROM afisha.purchases
    GROUP BY event_id
)
SELECT 
    *
FROM cnt
WHERE flag_rub = 1 AND flag_kzt = 1;

Running query in 'postgresql+psycopg2://praktikum_student@rc1e-oa2gp4mf9egcsq5k.mdb.yandexcloud.net:6432/data-analyst-afisha'

2 rows affected.

event_id,flag_rub,flag_kzt,rub_total,kzt_total
42299,1,1,936530.94,12570.89
546811,1,1,289819.94,1268766.9


**Результат:** 2 мероприятия из 22 484 (0.009%) принимали обе валюты: rub 936 530.94 / kzt 12 570.89 и rub 289 819.94 / kzt 1 268 766.90 — второе фактически тенге-событие (тенге-оборот в 4.4 раза больше рублёвого). Вывод: валюта — свойство мероприятия в 99.99% случаев; «не смешивать валюты в SUM» — обязательно, «валюта назначается на событие» — с оговоркой.

## Этап 14. Сводные выводы

1. **Схема подтверждена данными.** Пять таблиц, все связи 1:M; `order_id` уникален (292 034 = 292 034), `event_id` повторяется (≈13 заказов на мероприятие). Классическая звезда: факты в `purchases`, измерения в справочниках.
2. **Пропусков нет** — все 30 колонок пяти таблиц с `null_count = 0`. Проверка автоматизирована через `information_schema`.
3. **Категориальные данные чистые** — канонические значения без опечаток. Особенности: 2 валюты (rub 98.3% / kzt 1.7%), mobile ~80% заказов, 36 операторов, малые категории unstable (other — 2 заказа, tv — 3).
4. **Период:** июнь–октябрь 2024, 153 дня без пропусков, ~2 000 заказов/день, 22 000 пользователей, 81 регион. 
5. **Классификация мероприятий по выручке:** 22 123 платных + 295 бесплатных (доказано: у всех zero-заказов есть билеты; среди zero-мероприятий нет платных) + 66 мероприятий без единой продажи с отрицательным денежным потоком = 22 484. Счётное тождество сошлось. 
6. **Возвраты — строки-зеркала:** 381 заказ с `total < 0` и `revenue` = −тариф × |total|; при агрегации выручки отделять от продаж.
7. **Модель комиссии восстановлена полностью.** Тариф — целое число процентов из сетки 1–34% (24 значения: плотная зона 1–22% и экзотика 32/34%), назначается на уровне мероприятия и фиксирован у всех 22 123 мероприятий с продажами (после допуска на копеечный джиттер: 22 123 = 22 123); `revenue` = тариф × `total`, округлённый до копеек. Оператор коррелирует с тарифом (средние 3.5–12.9%), но не определяет его. Чистая лестница тарифов: рабочая зона 2–13% держит 94.3% заказов (вершина плоская: 4%, 9%, 8%, 3%, 11% — доли 8.8–12.1%, единого «стандартного» тарифа нет); премиум-хвост 14–17% — 5.4% заказов; экзотика 18–34% — ~440 заказов (0.16%). **Средневзвешенный тариф ≈ 7.9% — эффективная комиссия сервиса**; медианный заказ сидит на тарифе 11%. Счётное тождество сходится до заказа: 285 881 (продажи) = 292 034 (все) − 5 772 (бесплатные) − 381 (возвраты).
8. **Статистики чека (руб., продажи):** медиана 4 826 ₽, среднее 6 465 ₽, MAX 811 745 ₽ (168× медианы — экстремальные выбросы), MIN 0.93 ₽ (микро-аномалия). 
9. **Артефакты для выгрузки и дальнейшей аналитики:**
   - суммы только в разрезе `currency_code` (rub/kzt);
   - `event_id` ≠ мероприятие (873 дублей в справочнике);
   - города-омонимы — группировать по `city_id` (Глинополье ×2);
   - 66 мероприятий-призраков исключать из метрик продаж;
   - `total`/`revenue` в `double precision` — в отчётах приводить `::numeric`;
   - бесплатные (295) и возвраты (381) отделять от продаж.
10. **Валютная аудитория и валюта мероприятий:** 1 362 пользователя (6.19%) платят тенге — 5 067 продаж, ~3.7 заказа на человека против ~13 в среднем по базе; только 2 из 22 484 мероприятий (0.009%) принимали обе валюты, у одного тенге-оборот в 4.4 раза больше рублёвого.
12. **Сезонность:** рост монотонный от июня к октябрю — заказы 34 840 → 100 600 (×2.9), комиссия сервиса 17.9 млн → 52.3 млн ₽ (×2.9), средняя комиссия на заказ стабильна (~514–587 ₽, летний пик +13%). Осень «горячее» лета, но годовые выводы на 5 месяцах невозможны.
13. **Распределение по типам мероприятий:** концерты доминируют (39.6% заказов), далее театр (23.2%) и «другое» (22.6%); спорт (7.5%) и стендап (4.6%) — средняя лига.
14. **Уроки метода:** артефакты хранения и выборки маскируются под бизнес-разброс:
    - MIN(revenue) = −90.76 выглядел «копеечной комиссией», а оказался зеркалом возврата → агрегат скрывает структуру, смотри паспорт строк;
    - «5%» из выборки возвратов — лишь один тариф сетки → свойство выборки ≠ свойство совокупности;
    - строгое MIN = MAX давало 36% «фиксированных» тарифов при джиттере округления → вывод о постоянстве делай с допуском на способ хранения;
    - счётные тождества ловят скрытые классы: 66 мероприятий нашлись как остаток тождества.

